# recs_016 — D5 option 3: habit/session rerank on frozen pools (**killed**)

**Verdict:** best blend NDCG@10 **0.040** vs D1 **0.093** (overall and slice A). Same failure mode as D2–D4.

## D5 map (habit / session embeddings)

All options reuse the same USE vectors (`q_session`, `u_reviews`, `u_behavior`, `habit_fused`). They differ in **what part of the pipeline you replace**.

| Option | Notebook | What changes | Retrieval | Ranking | Status |
|--------|----------|--------------|-----------|---------|--------|
| **Shipped** | eval job | retrieve + rank | `two_tower_v1` → top-100 | D1 `heuristic_logpop_blend` | **live** |
| **3** | **this notebook** | **rank only** (retrieval unchanged) | `two_tower_v1` → top-100 *(same)* | USE session / habit / blend on that pool | **killed** |
| **1** | [`recs_017`](../retrieval/recs_017_eval_habit_session_retrieval.ipynb) | new retriever + rerank | **habit vector** → full catalog top-M | session reranks those M | in progress |
| **1b** | `recs_017` | new single-stage scorer | — | **fused query** sorts full catalog | in progress |
| **2** | `recs_017` + export job | new retriever, old ranker | **habit vector** → export top-100 | D1 on those pools | follow-on |
| **4–5** | — | retrain tower | new learned tower | new learned tower | **deferred** |

**Why the old table said “frozen”:** it meant “this notebook does not *change* retrieval” — not a different retriever. Same `two_tower_v1` top-100 as shipped; this notebook loads those pools from eval artifacts and only tries new rankers.

**Option 3 in one line:** keep `two_tower_v1` candidates; try beating D1 by reordering the pool with embedding dot-products (not log-pop, not cascade).

**Not option 3:** full-catalog habit retrieval, cascade, or pool export → see `recs_017`.

### Methods in this notebook

| Method | Score on pool |
|--------|----------------|
| `…_session_embed_v1` | dot(`q_session`, item) |
| `…_habit_fused_embed_v1` | dot(`habit_fused`, item) |
| `…_session_habit_blend_v1` | tuned `w·session + (1−w)·habit` |

### Habit vs behavior

| Vector | Meaning | Built with |
|--------|---------|------------|
| `q_session` | This review | `embed(query_text)` |
| `u_reviews` | Past review *text* | mean embed of `train_review_rows` texts |
| `u_behavior` | Past play *catalog* signal | playtime-weighted mean app vectors (**same as fusion_c**) |
| `habit_fused` | Long-term taste combo | `normalize(0.5·u_behavior + 0.5·u_reviews)` |

**Promotion bar:** beat D1 on val **NDCG@10 overall + slice A**. Tune blend `w` on `train_tune` only.

**Deferred:** two-tower retrain (options 4–5), collaborative filtering v2.

## Setup

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import (
    cohort_parquet_path,
    load_retrieval_pool_rows,
    load_retrieval_pools_jsonl,
)
from steam_review_ml.evaluation.heuristic_ranker import (
    METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND,
    DEFAULT_LOGPOP_BLEND_ALPHA,
    minmax_norm,
    pool_rerank_registry,
    rerank_scores_on_pool,
    score_logpop_blend,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    average_precision_at_k,
    hit_rate_at_k,
    load_eval_examples_from_parquet,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
)
from steam_review_ml.recommender.retrieve import (
    ContentRetriever,
    _mean_train_review_text_embedding,
    _weighted_mean_behavior_embedding,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "two_tower_v1"
K_FINAL = 10
MIN_REVIEW_CHARS = 30
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPLIT_SEED = 2027
TUNE_FRAC = 0.10
MAX_VAL_EXAMPLES: int | None = None

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
TRAIN_COHORT_PARQUET = cohort_parquet_path(REPO_ROOT / "artifacts/recs/eval_cache/train_ranker_v1")
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_COHORT_PARQUET = cohort_parquet_path(REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1")

W_GRID = tuple(i / 10.0 for i in range(0, 11))  # session weight in session+habit blend

for p in (TRAIN_POOLS_PARQUET, TRAIN_COHORT_PARQUET, VAL_JSONL, VAL_COHORT_PARQUET):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

print(f"W_GRID session weights: {W_GRID[:3]} ... {W_GRID[-3:]}")

W_GRID session weights: (0.0, 0.1, 0.2) ... (0.8, 0.9, 1.0)


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load pools, catalog, examples

In [2]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
if MAX_VAL_EXAMPLES is not None:
    val_pools = val_pools[: int(MAX_VAL_EXAMPLES)]

train_examples = load_eval_examples_from_parquet(TRAIN_COHORT_PARQUET)
val_examples = load_eval_examples_from_parquet(VAL_COHORT_PARQUET)
train_ex_by_idx = {i: ex for i, ex in enumerate(train_examples)}
val_ex_by_idx = {i: ex for i, ex in enumerate(val_examples)}

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
X = np.asarray(retriever.embedding_matrix, dtype=np.float32)

print(
    f"train pools={len(train_pools):,}  val pools={len(val_pools):,}  "
    f"catalog={len(app_ids):,}"
)

train pools=51,691  val pools=12,500  catalog=315


## train_tune split + vector helpers

In [3]:
def stratified_ex_idx_split(
    pools: list[dict[str, Any]], *, tune_frac: float, seed: int
) -> tuple[set[int], set[int]]:
    """
    Split example indices from pools into fit/train and tune/validation sets,
    stratified by the 'slice_name' field.

    Parameters
    ----------
    pools : list[dict[str, Any]]
        List of pool rows, each containing at least 'ex_idx' and 'slice_name'.
    tune_frac : float
        Fraction of each slice to allocate to the tuning (validation) set.
    seed : int
        Random seed for reproducibility.

    Returns
    -------
    fit_ids : set[int]
        Set of example indices for the fit (training) set.
    tune_ids : set[int]
        Set of example indices for the tuning (validation) set.

    Notes
    -----
    Always assigns at least one sample from each slice to the tuning set.
    """
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for ids in by_slice.values():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


def l2_normalize(v: np.ndarray) -> np.ndarray:
    arr = np.asarray(v, dtype=np.float32).ravel()
    nrm = float(np.linalg.norm(arr))
    if nrm <= 1e-12:
        return arr
    return (arr / nrm).astype(np.float32)


def build_user_vectors(ex: dict[str, Any]) -> dict[str, np.ndarray]:
    """fusion_c-aligned behavior + review habit vectors."""
    q_session = np.asarray(retriever.embed_text(str(ex["query_text"])), dtype=np.float32)
    u_reviews = _mean_train_review_text_embedding(retriever, ex)
    u_behavior, _ = _weighted_mean_behavior_embedding(
        ex,
        embedding_matrix=X,
        app_to_row=app_to_row,
        fallback=u_reviews,
    )
    habit_fused = l2_normalize(0.5 * u_behavior + 0.5 * u_reviews)
    return {
        "q_session": q_session,
        "u_reviews": u_reviews,
        "u_behavior": u_behavior,
        "habit_fused": habit_fused,
    }


def pool_dot_scores(pool_app_ids: list[int], q: np.ndarray) -> np.ndarray:
    rows = [app_to_row[int(a)] for a in pool_app_ids]
    return (X[np.asarray(rows, dtype=np.int64)] @ np.asarray(q, dtype=np.float32).ravel()).astype(np.float64)


fit_ex_idx, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]
print(f"train_tune={len(train_tune):,}")

train_tune=5,169


## Precompute user vectors (train_tune + val)

In [4]:
def vector_cache_for_pools(
    pools: list[dict[str, Any]],
    ex_by_idx: dict[int, dict[str, Any]],
) -> dict[int, dict[str, np.ndarray]]:
    out: dict[int, dict[str, np.ndarray]] = {}
    for i, row in enumerate(pools):
        ex_idx = int(row["ex_idx"])
        if ex_idx not in ex_by_idx:
            raise KeyError(f"missing example for ex_idx={ex_idx}")
        out[ex_idx] = build_user_vectors(ex_by_idx[ex_idx])
        if (i + 1) % 1000 == 0:
            print(f"  vectors {i + 1:,}/{len(pools):,}", flush=True)
    return out


print("Precomputing vectors on train_tune...")
vec_tune = vector_cache_for_pools(train_tune, train_ex_by_idx)
print("Precomputing vectors on val...")
vec_val = vector_cache_for_pools(val_pools, val_ex_by_idx)
print(f"cached tune={len(vec_tune):,}  val={len(vec_val):,}")

Precomputing vectors on train_tune...


2026-06-12 09:30:28.879396: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781271028.888840  120798 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781271028.892842  120798 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781271028.942238  120798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781271028.942261  120798 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781271028.942262  120798 computation_placer.cc:177] computation placer alr

  vectors 1,000/5,169
  vectors 2,000/5,169
  vectors 3,000/5,169
  vectors 4,000/5,169
  vectors 5,000/5,169
Precomputing vectors on val...
  vectors 1,000/12,500
  vectors 2,000/12,500
  vectors 3,000/12,500
  vectors 4,000/12,500
  vectors 5,000/12,500
  vectors 6,000/12,500
  vectors 7,000/12,500
  vectors 8,000/12,500
  vectors 9,000/12,500
  vectors 10,000/12,500
  vectors 11,000/12,500
  vectors 12,000/12,500
cached tune=5,169  val=12,500


## Scoring recipes on frozen pool

In [5]:
D1_SPEC = pool_rerank_registry()[METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND]


def positives_from_row(row: dict[str, Any]) -> set[int]:
    return set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))


def pool_apps_and_retr(row: dict[str, Any]) -> tuple[list[int], list[float]]:
    apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    retr = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    return apps, retr


def score_session_only(pool_apps: list[int], vecs: dict[str, np.ndarray]) -> np.ndarray:
    return minmax_norm(pool_dot_scores(pool_apps, vecs["q_session"]))


def score_habit_fused_only(pool_apps: list[int], vecs: dict[str, np.ndarray]) -> np.ndarray:
    return minmax_norm(pool_dot_scores(pool_apps, vecs["habit_fused"]))


def score_session_habit_blend(
    pool_apps: list[int], vecs: dict[str, np.ndarray], *, w_session: float
) -> np.ndarray:
    s = minmax_norm(pool_dot_scores(pool_apps, vecs["q_session"]))
    h = minmax_norm(pool_dot_scores(pool_apps, vecs["habit_fused"]))
    w = float(w_session)
    return w * s + (1.0 - w) * h


def score_d1(pool_apps: list[int], retr: list[float]) -> np.ndarray:
    return rerank_scores_on_pool(
        pool_apps, retr, D1_SPEC, pop_row=pop_row, app_to_row=app_to_row
    )


def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    k_final: int,
) -> np.ndarray:
    """Scatter pool scores into catalog space, then rank (same as recs_014/015)."""
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def mean_ndcg(
    pools: list[dict[str, Any]],
    vec_cache: dict[int, dict[str, np.ndarray]],
    score_fn: Callable[[list[int], list[float], dict[str, np.ndarray]], np.ndarray],
) -> float:
    scores: list[float] = []
    for row in pools:
        pool_apps, retr = pool_apps_and_retr(row)
        vecs = vec_cache[int(row["ex_idx"])]
        blend = score_fn(pool_apps, retr, vecs)
        positives = positives_from_row(row)
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        scores.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(scores)) if scores else 0.0


def _score_session(pool_apps, retr, vecs):
    return score_session_only(pool_apps, vecs)


def _score_habit(pool_apps, retr, vecs):
    return score_habit_fused_only(pool_apps, vecs)


def _score_d1(pool_apps, retr, vecs):
    return score_d1(pool_apps, retr)


print("scoring helpers ready")

scoring helpers ready


## Tune session/habit blend on train_tune

In [6]:
best_w = 0.5
best_tune_ndcg = -1.0
for w in W_GRID:
    def _blend(pool_apps, retr, vecs, w=w):
        return score_session_habit_blend(pool_apps, vecs, w_session=w)

    nd = mean_ndcg(train_tune, vec_tune, _blend)
    if nd > best_tune_ndcg:
        best_tune_ndcg = nd
        best_w = float(w)

print(f"best_w_session={best_w:.1f}  train_tune NDCG@{K_FINAL}={best_tune_ndcg:.4f}")

best_w_session=0.0  train_tune NDCG@10=0.0506


## Val head-to-head vs D1

In [7]:
METHOD_SESSION = f"{POOL_METHOD}_ranker_d5_session_embed_v1"
METHOD_HABIT = f"{POOL_METHOD}_ranker_d5_habit_fused_embed_v1"
METHOD_BLEND = f"{POOL_METHOD}_ranker_d5_session_habit_blend_v1"


def eval_pool_row(
    row: dict[str, Any],
    vec_cache: dict[int, dict[str, np.ndarray]],
    method: str,
) -> dict[str, Any]:
    pool_apps, retr = pool_apps_and_retr(row)
    vecs = vec_cache[int(row["ex_idx"])]
    if method == METHOD_SESSION:
        scores = score_session_only(pool_apps, vecs)
    elif method == METHOD_HABIT:
        scores = score_habit_fused_only(pool_apps, vecs)
    elif method == METHOD_BLEND:
        scores = score_session_habit_blend(pool_apps, vecs, w_session=best_w)
    elif method == METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND:
        scores = score_d1(pool_apps, retr)
    elif method == POOL_METHOD:
        scores = np.asarray(retr, dtype=np.float64)
    else:
        raise ValueError(method)
    positives = positives_from_row(row)
    ranked = pool_scores_to_ranked_indices(pool_apps, scores, k_final=K_FINAL)
    return {
        "method": method,
        "slice_name": str(row["slice_name"]),
        "ex_idx": int(row["ex_idx"]),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
    }


methods = [
    POOL_METHOD,
    METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND,
    METHOD_SESSION,
    METHOD_HABIT,
    METHOD_BLEND,
]

val_rows = [
    eval_pool_row(row, vec_val, m)
    for row in val_pools
    for m in methods
]
df_val = pd.DataFrame(val_rows)
print("methods:", methods)

methods: ['two_tower_v1', 'two_tower_v1_heuristic_logpop_blend', 'two_tower_v1_ranker_d5_session_embed_v1', 'two_tower_v1_ranker_d5_habit_fused_embed_v1', 'two_tower_v1_ranker_d5_session_habit_blend_v1']


## Results

In [8]:
display(Markdown("### Val overall"))
display(
    df_val.groupby("method")[["Hit@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values("NDCG@K", ascending=False)
)

display(Markdown("### Val by slice"))
display(
    df_val.groupby(["slice_name", "method"])[["Hit@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values(["slice_name", "NDCG@K"], ascending=[True, False])
)

d1_ndcg = float(
    df_val.loc[df_val["method"] == METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "NDCG@K"].mean()
)
print(f"\nD1 val NDCG@{K_FINAL}={d1_ndcg:.4f}  (promotion bar)")
print(f"Blend tuned w_session={best_w:.1f}")
for m in (METHOD_SESSION, METHOD_HABIT, METHOD_BLEND):
    nd = float(df_val.loc[df_val["method"] == m, "NDCG@K"].mean())
    print(f"  {m}: NDCG@{K_FINAL}={nd:.4f} ({'BEATS D1' if nd > d1_ndcg else 'below D1'})")

### Val overall

,Hit@K,NDCG@K,MRR
method,,,
two_tower_v1_heuristic_logpop_blend,0.19328,0.092892,0.067059
two_tower_v1_ranker_d5_session_habit_blend_v1,0.08912,0.039637,0.027956
two_tower_v1_ranker_d5_habit_fused_embed_v1,0.08912,0.039637,0.027956
two_tower_v1_ranker_d5_session_embed_v1,0.07952,0.038236,0.028755
two_tower_v1,0.04680,0.018161,0.011008


### Val by slice

Hit@K  \
slice_name            method                                                    
slice_a_multi_target  two_tower_v1_heuristic_logpop_blend            0.271724   
                      two_tower_v1_ranker_d5_habit_fused_embed_v1    0.180690   
                      two_tower_v1_ranker_d5_session_habit_blend_v1  0.180690   
                      two_tower_v1_ranker_d5_session_embed_v1        0.153103   
                      two_tower_v1                                   0.091034   
slice_b_single_target two_tower_v1_heuristic_logpop_blend            0.188450   
                      two_tower_v1_ranker_d5_habit_fused_embed_v1    0.083482   
                      two_tower_v1_ranker_d5_session_habit_blend_v1  0.083482   
                      two_tower_v1_ranker_d5_session_embed_v1        0.074989   
                      two_tower_v1                                   0.044076   

                                                                       NDCG@K  \
slice_name            method                                                    
slice_a_multi_target  two_tower_v1_heuristic_logpop_blend            0.068322   
                      two_tower_v1_ranker_d5_habit_fused_embed_v1    0.046758   
                      two_tower_v1_ranker_d5_session_habit_blend_v1  0.046758   
                      two_tower_v1_ranker_d5_session_embed_v1        0.041195   
                      two_tower_v1                                   0.020537   
slice_b_single_target two_tower_v1_heuristic_logpop_blend            0.094404   
                      two_tower_v1_ranker_d5_habit_fused_embed_v1    0.039198   
                      two_tower_v1_ranker_d5_session_habit_blend_v1  0.039198   
                      two_tower_v1_ranker_d5_session_embed_v1        0.038054   
                      two_tower_v1                                   0.018015   

                                                                          MRR  
slice_name            method                                                   
slice_a_multi_target  two_tower_v1_heuristic_logpop_blend            0.081396  
                      two_tower_v1_ranker_d5_habit_fused_embed_v1    0.059895  
                      two_tower_v1_ranker_d5_session_habit_blend_v1  0.059895  
                      two_tower_v1_ranker_d5_session_embed_v1        0.056281  
                      two_tower_v1                                   0.021192  
slice_b_single_target two_tower_v1_heuristic_logpop_blend            0.066176  
                      two_tower_v1_ranker_d5_habit_fused_embed_v1    0.025990  
                      two_tower_v1_ranker_d5_session_habit_blend_v1  0.025990  
                      two_tower_v1_ranker_d5_session_embed_v1        0.027060  
                      two_tower_v1                                   0.010381


D1 val NDCG@10=0.0929  (promotion bar)
Blend tuned w_session=0.0
  two_tower_v1_ranker_d5_session_embed_v1: NDCG@10=0.0382 (below D1)
  two_tower_v1_ranker_d5_habit_fused_embed_v1: NDCG@10=0.0396 (below D1)
  two_tower_v1_ranker_d5_session_habit_blend_v1: NDCG@10=0.0396 (below D1)


### Takeaway

- **Option 3 killed** — session, habit, and tuned blend all below D1 on val (best ~0.040 NDCG@10 vs D1 0.093). Same log-pop barrier as D2–D4.
- **Next:** `recs_017` options 1 / 1b / 2 (change retrieval, not pool rerank only).
- Options 4–5 (two-tower retrain) and CF v2 remain deferred.